In [67]:
import torch
import torch.nn as nn

torch.manual_seed(123)
torch.set_printoptions(sci_mode=False)

In [68]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits



In [69]:
model = NeuralNetwork(50, 3)

In [70]:
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [71]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
num_params

2213

In [72]:
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

In [73]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [74]:
model.layers[0].bias

Parameter containing:
tensor([-0.1250,  0.0513,  0.0366,  0.0075,  0.0509,  0.0545, -0.0393,  0.0924,
        -0.1412, -0.1232, -0.1063,  0.0081, -0.1249,  0.0101, -0.0019, -0.1298,
         0.1388, -0.0330,  0.1017,  0.1247, -0.0554, -0.0417,  0.1388,  0.0159,
         0.1215,  0.0385,  0.0769, -0.1224, -0.0279,  0.0991],
       requires_grad=True)

In [75]:
X = torch.rand(1, 50)
out = model(X)
out


tensor([[-0.1670,  0.1001, -0.1219]], grad_fn=<AddmmBackward0>)

In [76]:
with torch.no_grad():
    out = model(X)
out

tensor([[-0.1670,  0.1001, -0.1219]])

In [77]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=-1)
out

tensor([[0.2983, 0.3896, 0.3121]])

In [78]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [79]:
X_train.shape

torch.Size([5, 2])

In [80]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [81]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [82]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.7000, -1.5000],
        [ 2.3000, -1.1000]]) tensor([1, 1])
Batch 2: tensor([[-0.9000,  2.9000],
        [-1.2000,  3.1000]]) tensor([0, 0])


In [83]:
import torch.nn.functional as F


model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        ###Logging
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

model.eval()

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.64
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.24
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.10
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.05
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.02
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.01


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)

In [84]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
num_params

752

In [85]:
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 1.6612, -3.2363],
        [ 1.5880, -3.0690],
        [ 1.4475, -2.7835],
        [-2.1527,  1.9212],
        [-2.5079,  2.2423]])


In [86]:
probas = torch.softmax(outputs, dim=-1)
probas

tensor([[0.9926, 0.0074],
        [0.9906, 0.0094],
        [0.9857, 0.0143],
        [0.0167, 0.9833],
        [0.0086, 0.9914]])

In [88]:
predictions = torch.argmax(probas, dim=-1)
predictions

tensor([0, 0, 0, 1, 1])

In [89]:
predictions == y_train

tensor([True, True, True, True, True])

In [90]:
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=-1)
        compare = labels == predictions

        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [92]:
compute_accuracy(model, test_loader)

1.0

In [93]:
torch.save(model.state_dict(), 'mlp.pth')

In [94]:
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load('mlp.pth'))

<All keys matched successfully>

In [95]:
torch.mps.is_available()

True

In [96]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])

In [97]:
tensor_1 = tensor_1.to("mps")
tensor_2 = tensor_2.to("mps")
print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='mps:0')


In [99]:
model = NeuralNetwork(2, 2)

device = torch.device('mps')
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
model.eval()

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.78
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.63
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.27
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.40
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.09
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.03


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)